# 02. Teacher LLM In-Context Labeling & Multi-Path Reasoning
## Deng et al. (WWW '23 Companion) Section 3.1

This notebook demonstrates:
1. Loading the 6-shot demonstration prompt with Chain-of-Thought financial reasoning.
2. Generating $K=8$ independent stochastic reasoning paths at temperature $T=0.5$.
3. Aggregating via majority voting and calculating continuous soft agreement scores.
4. Generating `weak_labels.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.llm.client import get_llm_provider
from src.llm.prompts import PromptManager
from src.llm.labeling import generate_teacher_weak_labels_dataset
from src.llm.aggregation import aggregate_paths_dataframe, save_weak_labels
from src.utils.common import DATA_PROC

In [ ]:
# Load small sample of train split for labeling demonstration
train_df = pd.read_parquet(DATA_PROC / "train.parquet").head(10)
print(f"Loaded {len(train_df)} posts for labeling.")

provider = get_llm_provider()
prompt_mgr = PromptManager("sentiment_prompt")

# Generate 8 reasoning paths per post
paths_df = generate_teacher_weak_labels_dataset(
    df=train_df,
    provider=provider,
    prompt_manager=prompt_mgr,
    num_paths=8,
    include_cot=True,
)
paths_df.head(16)

In [ ]:
# Aggregate multi-path predictions
weak_df = aggregate_paths_dataframe(paths_df, original_df=train_df)
print("Weak Labels with Soft Scores:")
weak_df[["post_id", "ticker", "dominant_label", "agreement_score", "teacher_soft_score"]]